In [1]:
import os
import pathlib
import datetime as dt
from typing import Union
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Load environment profiles up from parent directory
load_dotenv(dotenv_path="../.env")

# Map target paths safely via environment drivers
RAW_DIR = pathlib.Path(os.getenv("DATA_DIR_RAW", "data/raw"))
PROC_DIR = pathlib.Path(os.getenv("DATA_DIR_PROCESSED", "data/processed"))

# Resolve paths relative to the project root directory
# (Going up one level from the notebooks directory)
PROJECT_ROOT = pathlib.Path.cwd().parent
RAW_PATH = (PROJECT_ROOT / RAW_DIR).resolve()
PROC_PATH = (PROJECT_ROOT / PROC_DIR).resolve()

# Ensure targeted folders are established cleanly
RAW_PATH.mkdir(parents=True, exist_ok=True)
PROC_PATH.mkdir(parents=True, exist_ok=True)

print("Targeted Raw Storage Directory:", RAW_PATH)
print("Targeted Processed Storage Directory:", PROC_PATH)

# Dynamic file-stamping tool
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M')

Targeted Raw Storage Directory: C:\Users\imema\bootcamp_mattia_imeri\homework\homework05\data\raw
Targeted Processed Storage Directory: C:\Users\imema\bootcamp_mattia_imeri\homework\homework05\data\processed


In [2]:
# Secure reproduction parity via generator seeding
np.random.seed(42)

trading_dates = pd.date_range("2026-01-01", periods=15, freq="D")
df_market = pd.DataFrame({
    'date': trading_dates,
    'ticker': ['AAPL'] * 15,
    'price': 180.0 + np.random.randn(15).cumsum()
})

print("--- Sample Portfolio Dataset Instantiated ---")
df_market.info()
display(df_market.head(3))

--- Sample Portfolio Dataset Instantiated ---
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    15 non-null     datetime64[us]
 1   ticker  15 non-null     str           
 2   price   15 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 552.0 bytes


,date,ticker,price
0,2026-01-01,AAPL,180.496714
1,2026-01-02,AAPL,180.358450
2,2026-01-03,AAPL,181.006138


In [5]:
def detect_format(path: Union[str, pathlib.Path]) -> str:
    suffix = str(path).lower()
    if suffix.endswith('.csv'): 
        return 'csv'
    if suffix.endswith('.parquet') or suffix.endswith('.pq') or suffix.endswith('.parq'): 
        return 'parquet'
    raise ValueError(f"Unsupported file format provided: {path}")

def write_df(df: pd.DataFrame, path: Union[str, pathlib.Path]) -> pathlib.Path:
    path = pathlib.Path(path)
    # Handle missing directories safely on write executions
    path.parent.mkdir(parents=True, exist_ok=True)
    
    fmt = detect_format(path)
    if fmt == 'csv':
        df.to_csv(path, index=False)
    elif fmt == 'parquet':
        try:
            df.to_parquet(path)
        except Exception as e:
            raise RuntimeError(
                "Parquet storage operation rejected. Dependency Engine Missing. "
                "Please execute 'pip install pyarrow' inside your active environment context."
            ) from e
    return path

def read_df(path: Union[str, pathlib.Path]) -> pd.DataFrame:
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Requested target file path does not exist: {path}")
        
    fmt = detect_format(path)
    if fmt == 'csv':
        # Check for column presence to auto-parse date types correctly if relevant
        preview = pd.read_csv(path, nrows=0)
        if 'date' in preview.columns:
            return pd.read_csv(path, parse_dates=['date'])
        return pd.read_csv(path)
    elif fmt == 'parquet':
        try:
            return pd.read_parquet(path)
        except Exception as e:
            raise RuntimeError(
                "Parquet retrieval engine failed. Verify PyArrow install configuration."
            ) from e

In [6]:
stamp = ts()

# Generate custom filenames matching your delivery requirements
csv_target = RAW_PATH / f"sample_{stamp}.csv"
parquet_target = PROC_PATH / f"sample_{stamp}.parquet"

# Route through your new custom utilities
write_df(df_market, csv_target)
print(f"Raw file committed safely via env paths -> {csv_target}")

try:
    write_df(df_market, parquet_target)
    print(f"Processed Parquet committed safely via env paths -> {parquet_target}")
except RuntimeError as e:
    print(e)

Raw file committed safely via env paths -> C:\Users\imema\bootcamp_mattia_imeri\homework\homework05\data\raw\sample_20260817-2149.csv
Processed Parquet committed safely via env paths -> C:\Users\imema\bootcamp_mattia_imeri\homework\homework05\data\processed\sample_20260817-2149.parquet


In [7]:
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame, target_cols=('date', 'ticker', 'price')) -> dict:
    checks = {
        'shape_parity_achieved': original.shape == reloaded.shape,
        'expected_columns_present': all(col in reloaded.columns for col in target_cols)
    }
    if 'price' in reloaded.columns:
        checks['price_type_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    if 'date' in reloaded.columns:
        checks['date_type_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    return checks

# Execute reload workflows
df_reloaded_csv = read_df(csv_target)
print("Evaluating Reloaded CSV Parity metrics:")
print(validate_loaded(df_market, df_reloaded_csv))

if parquet_target.exists():
    df_reloaded_parquet = read_df(parquet_target)
    print("\nEvaluating Reloaded Parquet Parity metrics:")
    print(validate_loaded(df_market, df_reloaded_parquet))

Evaluating Reloaded CSV Parity metrics:
{'shape_parity_achieved': True, 'expected_columns_present': True, 'price_type_numeric': True, 'date_type_datetime': True}

Evaluating Reloaded Parquet Parity metrics:
{'shape_parity_achieved': True, 'expected_columns_present': True, 'price_type_numeric': True, 'date_type_datetime': True}
